---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI and Data-Driven Marketplaces

### 📋 **Topic**: Randomized Assignment and Balance Tests

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---


> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/ma/blob/main/assets/before-you-begin.md)

## Overview

Let's use our Python knowledge to perform a randomized assignment, and verify we did it correctly.

**What we'll learn:**
- How to perform randomized assignment
- How to check if randomization worked
- Balance tests and their importance
- Setting random seeds for reproducibility


In [ ]:
# Let's import the libraries we will use
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Locate the course folder from either ma/ or ma/scripts/.
course_folder = Path.cwd()
if not (course_folder / "data").is_dir():
    course_folder = course_folder.parent

data_folder = course_folder / "data"
output_folder = course_folder / "temp"
output_folder.mkdir(parents=True, exist_ok=True)

# Load user data for randomization.
df_users = pd.read_csv(data_folder / "users.csv")

print("Dataset loaded successfully!")
print(f"Dataset shape: {df_users.shape}")
print(f"Columns: {df_users.columns.tolist()}")

print("Sample of user data:")
print(df_users.head())


## Randomized Assignment with a 20% Probability

Each user has a 20% chance of receiving treatment, independently of the other users. We expect roughly 20% in treatment and 80% in control, but the exact numbers can vary.


In [ ]:
# A random number between 0 and 50. Run this cell a few times.
np.random.rand()*50

In [ ]:
# The seed lets us reproduce the same assignment.
np.random.seed(44)
# Draw a number between 0 and 1 for each user; numbers below 0.2 receive treatment.
df_users["treatment"] = np.where(np.random.rand(len(df_users)) < 0.2, "Treatment", "Control")

df_assignment = (
    df_users.groupby("treatment")["user"].count().reset_index(name="n")
)
print(df_assignment)

In [ ]:
df_users

## Exact Randomized Assignment

We can also fix the group sizes in advance. Here, we randomly choose half the users for treatment and put the rest in control. Who receives treatment is random; how many receive it is fixed. With an odd number of users, control gets one extra user.

We'll use a separate copy, `df_users_exact`, so we keep our original assignment for the balance checks below.

In [ ]:
# Work on a copy so we do not change the original assignment.
df_users_exact = df_users.copy()

# Fix the number of treated users. // divides and rounds down.
n_treatment = len(df_users_exact) // 2

# Choose that many users at random, without selecting anyone twice.
# random_state makes this selection reproducible.
treatment_indices = df_users_exact.sample(n=n_treatment, replace=False, random_state=44).index

# Start everyone in control, then assign the selected users to treatment.
df_users_exact["treatment"] = "Control"
df_users_exact.loc[treatment_indices, "treatment"] = "Treatment"

# Check that the group sizes match our plan.
df_exact_assignment = df_users_exact.groupby("treatment").size().reset_index(name="n")
print(df_exact_assignment)

Exact group sizes do not guarantee identical ages, genders, or earnings in the two groups. Those characteristics can still differ by chance.

For the rest of the notebook, we'll return to `df_users`, our original assignment with a 20% treatment probability.

## Balance Tests

If randomized assignment was performed correctly, then the treatment groups should be similar with respect to the attributes we can observe. Let's test this:


In [ ]:
# -----------------------------------------------
# Gender balance test
# -----------------------------------------------

# 1. Group data by treatment status (Treatment vs Control)
#    and gender, then count how many users are in each group.
#    `.size()` is a simple way to count rows.
df_balance_gender = (
    df_users.groupby(["treatment", "gender"])
    .size()                         # counts number of rows per (treatment, gender)
    .unstack(fill_value=0)          # turns 'gender' values into separate columns
)

# 2. Add a "Total" column to show total users in each treatment group
df_balance_gender["Total"] = df_balance_gender.sum(axis=1)

# 3. Calculate percentages within each treatment group
#    (divide each count by the row total, multiply by 100)
df_balance_gender_pct = (
    df_balance_gender.div(df_balance_gender["Total"], axis=0) * 100
).round(3)  # round to one decimal place

# 4. Print the results
print("Gender balance (counts):")
print(df_balance_gender)

print("\nGender balance (% within treatment):")
print(df_balance_gender_pct)


In [ ]:
# -----------------------------------------------
# Earnings balance test
# -----------------------------------------------

# 1. Group the data by treatment status (Treatment vs Control)
#    and calculate basic summary numbers for earnings:
#    - average earnings (mean)
#    - variation in earnings (standard deviation)
#    - how many users are in each group (count)
df_balance_earnings = (
    df_users.groupby("treatment")
    .agg({"earnings": ["mean", "std", "count"]})  # aggregate multiple stats at once
    .round(2)                                     # round numbers to 2 decimal places
)

# 2. Flatten the multi-level column names that result from using .agg()
#    so instead of ("earnings", "mean") we just have "avg_earnings", etc.
df_balance_earnings.columns = ["avg_earnings", "std_earnings", "count"]

# 3. Move "treatment" from the index back into a normal column
df_balance_earnings = df_balance_earnings.reset_index()

# 4. Print the summary table
print("Earnings balance test:")
print(df_balance_earnings)


In [ ]:
# -----------------------------------------------
# Age balance visualization
# -----------------------------------------------

# 1. Filter out users with missing or invalid ages (e.g., 0 or NaN)
#    This ensures we only plot meaningful age values.
df_users_age = df_users[df_users["age"] > 0].copy()

# 1. Create a new figure and set its size (in inches)
plt.figure(figsize=(10, 6))

# 2. Draw overlapping histograms for each treatment group
#    - data: our user dataset filtered for valid ages
#    - x="age": the variable we're plotting
#    - hue="treatment": separate colors for Treatment vs Control
#    - alpha=0.6: transparency so both histograms are visible
#    - bins=30: number of histogram bars
#    - kde=True: adds a smooth curve showing the density
sns.histplot(
    data=df_users_age,
    x="age",
    hue="treatment",
    hue_order=["Control", "Treatment"],
    palette={"Control": "#1f77b4", "Treatment": "#ff7f0e"},
    bins=30,
    alpha=0.6,
    kde=True
)

# 3. Add axis labels and a descriptive title
plt.xlabel("Age", fontsize=12)
plt.ylabel("Number of Users", fontsize=12)
plt.title("Age Distribution by Treatment Group", fontsize=14)

# 4. Move Seaborn's legend without losing the group labels
sns.move_legend(plt.gca(), "upper right", title="Group")

# 5. Add a light grid for readability
plt.grid(True, alpha=0.3)

# 6. Adjust spacing so labels and titles fit nicely
plt.tight_layout()

# 7. Save the figure to a PDF file (high resolution)
plt.savefig(output_folder / "age_balance_test.pdf", dpi=300, bbox_inches="tight")

# (Optional) Close the plot to free up memory in batch scripts
# plt.close()

print("✅ Age balance plot saved to 'temp/age_balance_test.pdf'")


### Compare percentages, not counts

The control group has more users, so its bars are taller. That does not mean its users are older or younger.

Let's show the same data as percentages within each group. Each group's bars now add up to 100%, so we can compare the shapes rather than the group sizes. We keep the same users, assignments, and age bins. Both visualizations include only users with a recorded, positive age.

Similar distributions are a useful balance check, not proof that randomization was performed correctly.


In [ ]:
# Compare the age distributions on the same scale.
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(
    data=df_users_age,
    x="age",
    hue="treatment",
    hue_order=["Control", "Treatment"],
    palette={"Control": "#1f77b4", "Treatment": "#ff7f0e"},
    bins=30,
    common_bins=True,       # Use the same age bins for both groups.
    stat="percent",        # Show percentages instead of counts.
    common_norm=False,      # Make each group's bars add up to 100%.
    alpha=0.35,
    kde=True,
    ax=ax,
)

ax.set_xlabel("Age", fontsize=12)
ax.set_ylabel("Users within each group (%)", fontsize=12)
ax.set_title("Age Distribution by Treatment Group (Normalized)", fontsize=14, pad=12)
sns.move_legend(ax, "upper right", title="Group")
ax.grid(True, alpha=0.3)
fig.text(0.02, 0.02, "Includes only users with a recorded, positive age.", fontsize=10, color="0.4")
fig.tight_layout(rect=(0, 0.05, 1, 1))

# Save this version separately so we keep the count-based visualization too.
fig.savefig(output_folder / "age_balance_normalized.pdf", dpi=300, bbox_inches="tight")
plt.show()
